# Pipeline Chunking & Seed Generation — Sirah Nabawiyah

Notebook ini menggabungkan dua tahap:
1. **Chunking** — Memecah dokumen hasil preprocessing menjadi chunk ≤1500 karakter dengan overlap 1 kalimat
2. **Seed Generation** — Sampling stratified per bab (maks 25 chunk/bab) untuk template anotasi NER manual

**Output:**
- `sirah_chunks_final.csv` — Seluruh chunk
- `sirah_manual_seed.csv` — Sampel chunk + kolom anotasi kosong (PERSON / LOCATION / EVENT / TIME)

---
## Part 1: Chunking
### 1.1 Import & Konfigurasi

In [1]:
import re
import pandas as pd
from pathlib import Path

# ── Path ──────────────────────────────────────────────────────────────
IN_CSV     = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\preprocessing_result\sirah_simple_clean.csv")
OUT_CHUNKS = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\chunking_result\sirah_chunks_final.csv")
OUT_SEED   = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_manual_seed.csv")

# ── Parameter Chunking ────────────────────────────────────────────────
MAX_CHARS      = 1500   # batas maks karakter per chunk
OVERLAP_SENTS  = 1      # jumlah kalimat overlap antar chunk

# ── Parameter Sampling ────────────────────────────────────────────────
PER_BAB        = 25     # maks chunk per bab untuk seed anotasi
RANDOM_STATE   = 42     # reproducibility

### 1.2 Baca Data Hasil Preprocessing

In [2]:
df = pd.read_csv(IN_CSV, sep=";", encoding="utf-8-sig").fillna("")
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

if "teks_clean" not in df.columns:
    raise ValueError(f"Kolom 'teks_clean' tidak ditemukan. Kolom yang ada: {df.columns.tolist()}")

print("Rows dokumen:", len(df))
df.head(3)

Rows dokumen: 389


,judul_bab,judul_sub_bab,halaman,teks_clean
0,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
1,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
2,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...


### 1.3 Fungsi Sentence Splitting & Chunking

In [3]:
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text: str) -> list[str]:
    """Split teks menjadi list kalimat berdasarkan tanda baca akhir."""
    text = (text or "").strip()
    if not text:
        return []
    t = re.sub(r"\s+", " ", text).strip()
    sents = [s.strip() for s in _SENT_SPLIT.split(t) if s.strip()]

    # Fallback: jika teks tidak punya tanda baca (OCR buruk),
    # jadikan seluruh teks sebagai 1 "kalimat"
    if not sents and t:
        sents = [t]

    return sents

def chunk_sentences(sentences: list[str],
                    max_chars: int = MAX_CHARS,
                    overlap_sents: int = OVERLAP_SENTS) -> list[str]:
    
    if not sentences:
        return []

    chunks = []
    buf = []  # buffer kalimat

    def buf_len(b: list[str]) -> int:
        return len(" ".join(b)) if b else 0

    for s in sentences:
        s = s.strip()
        if not s:
            continue

        # Cek apakah menambah kalimat ini melebihi max_chars
        if buf and (buf_len(buf) + 1 + len(s)) > max_chars:
            # Tutup chunk saat ini
            chunks.append(" ".join(buf).strip())

            # Overlap: bawa N kalimat terakhir ke chunk berikutnya
            if overlap_sents > 0:
                buf = buf[-overlap_sents:]
            else:
                buf = []

        # Kalimat tunggal > max_chars → langsung masuk sebagai chunk sendiri
        if not buf and len(s) > max_chars:
            chunks.append(s)
            continue

        buf.append(s)

    # Sisa buffer → chunk terakhir
    if buf:
        chunks.append(" ".join(buf).strip())

    return [c for c in chunks if c]


def make_chunk_id(doc_id: int, chunk_index: int) -> str:
    """Format: 000000-001"""
    return f"{doc_id:06d}-{chunk_index:03d}"

### 1.4 Jalankan Chunking & Simpan

In [4]:
rows = []
total_empty = 0

for doc_id, r in df.iterrows():
    text = (r["teks_clean"] or "").strip()
    if not text:
        total_empty += 1
        continue

    sents = split_sentences(text)
    chunks = chunk_sentences(sents, max_chars=MAX_CHARS, overlap_sents=OVERLAP_SENTS)

    if not chunks:
        total_empty += 1
        continue

    for i, ch in enumerate(chunks, start=1):
        rows.append({
            "chunk_id":      make_chunk_id(doc_id, i),
            "doc_id":        doc_id,
            "chunk_index":   i,
            "judul_bab":     r["judul_bab"],
            "judul_sub_bab": r["judul_sub_bab"],
            "halaman":       r["halaman"],
            "teks_chunk":    ch,
        })

df_chunks = pd.DataFrame(rows)

# Statistik chunk
df_chunks["len"] = df_chunks["teks_chunk"].str.len()
print(f"Input rows       : {len(df)}")
print(f"Dokumen kosong   : {total_empty}")
print(f"Total chunks     : {len(df_chunks)}")
print(f"Chunks > {MAX_CHARS} chars: {(df_chunks['len'] > MAX_CHARS).sum()}")
print(f"\nStatistik panjang chunk:")
print(df_chunks["len"].describe())

# Simpan
OUT_CHUNKS.parent.mkdir(parents=True, exist_ok=True)
df_chunks.drop(columns=["len"], errors="ignore").to_csv(
    OUT_CHUNKS, index=False, sep=";", encoding="utf-8-sig"
)
print(f"\nSaved: {OUT_CHUNKS}")
df_chunks.head(5)

Input rows       : 389
Dokumen kosong   : 0
Total chunks     : 1094
Chunks > 1500 chars: 4

Statistik panjang chunk:
count    1094.000000
mean     1205.526508
std       357.619857
min       104.000000
25%      1054.000000
50%      1377.000000
75%      1451.000000
max      2645.000000
Name: len, dtype: float64

Saved: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\chunking_result\sirah_chunks_final.csv


,chunk_id,doc_id,chunk_index,judul_bab,judul_sub_bab,halaman,teks_chunk,len
0,000000-001,0,1,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...,572
1,000001-001,1,1,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan...",1466
2,000001-002,1,2,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,Sekalipun begitu mereka tetap hidup berdamping...,910
3,000002-001,2,1,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...,959
4,000002-002,2,2,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,"Thayyi', Madzhij, Kindah, Lakham, Judzam, Uzd,...",1494



## Part 2: Seed Generation (Persiapan Manual Labelling)
### 2.1 Sampling Stratified & Ekspor Template Anotasi

In [5]:
dfc = df_chunks.drop(columns=["len"], errors="ignore").copy()

# Stratified sampling: maks PER_BAB chunk per bab
# group_keys=True + include_groups=False → judul_bab tetap ada via reset_index
seed = (
    dfc.groupby("judul_bab", group_keys=True)
       .apply(lambda g: g.sample(n=min(len(g), PER_BAB), random_state=RANDOM_STATE),
              include_groups=False)
       .reset_index(level=0)
       .reset_index(drop=True)
)

# Kolom inti + kolom anotasi manual (kosong)
cols = ["chunk_id", "doc_id", "chunk_index", "judul_bab", "judul_sub_bab", "halaman", "teks_chunk"]
seed_out = seed[cols].copy()
seed_out["entity_text"] = ""     # teks entitas persis seperti di teks_chunk
seed_out["label"]       = ""     # PERSON / LOCATION / EVENT / TIME
seed_out["notes"]       = ""     # catatan opsional
seed_out["start_char"]  = ""     # indeks awal
seed_out["end_char"]    = ""     # indeks akhir

# Simpan
OUT_SEED.parent.mkdir(parents=True, exist_ok=True)
seed_out.to_csv(OUT_SEED, index=False, sep=";", encoding="utf-8-sig")

print(f"Seed size        : {len(seed)}")
print(f"Maks chunk/bab   : {PER_BAB}")
print(f"Saved: {OUT_SEED}")
print(f"\nTop 10 bab:")
seed[["judul_bab"]].value_counts().head(10)

Seed size        : 844
Maks chunk/bab   : 25
Saved: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_manual_seed.csv

Top 10 bab:


judul_bab                                             
KELAHIRAN DAN EMPAT PULUH TAHUN SEBELUM NUBUWAH           25
KONSPIRASI UNTUK MEMBUNUH NABI                            25
KORESPONDENSI DENGAN BEBERAPA RAJA DAN AMIR               25
MANUSIA MEMASUKI AGAMA ALLAH SECARA BERBONDONG-BONDONG    25
PERANG AHZAB ATAU KHANDAQ                                 25
PERANG BADR KUBRA                                         25
PERANG DAN PENAKLUKAN MAKKAH                              25
PERANG HUNAIN                                             25
PERANG KHAIBAR DAN WADIL QURA                             25
PERANG TABUK                                              25
Name: count, dtype: int64

### 2.2 Statistik Coverage Manual Labelling

In [ ]:
df_all_chunks  = pd.read_csv(OUT_CHUNKS, sep=";", encoding="utf-8-sig")
df_prelabelled = pd.read_csv(
    Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_prelabelled.csv"),
    sep=";", encoding="utf-8-sig"
)

total_chunks   = df_all_chunks["chunk_id"].nunique()
seed_ids       = set(df_prelabelled["chunk_id"].unique())

n_labelled     = len(seed_ids)                  # chunk yang di-manual labelling
n_unlabelled   = total_chunks - n_labelled      # chunk yang tidak di-labelling

pct_labelled   = n_labelled / total_chunks * 100
pct_unlabelled = n_unlabelled / total_chunks * 100

print("=" * 55)
print("   STATISTIK COVERAGE MANUAL LABELLING")
print("=" * 55)
print(f"  Total chunk keseluruhan         : {total_chunks:>6}")
print(f"  Chunk yang di manual-labelling  : {n_labelled:>6}  ({pct_labelled:.1f}%)")
print(f"  Chunk TIDAK di manual-labelling : {n_unlabelled:>6}  ({pct_unlabelled:.1f}%)")
print("=" * 55)

   STATISTIK COVERAGE MANUAL LABELLING
  Total chunk keseluruhan         :   1094
  Chunk yang di manual-labelling  :    844  (77.1%)
  Chunk TIDAK di manual-labelling :    250  (22.9%)
